# Day 13: Beginner Attention, Transformers, and BERT Lab

This lab explains attention and transformers with small beginner-friendly examples.

Goal: understand the basic idea of attention before using large transformer models.

## 1. Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models

## 2. Attention Idea with Simple Numbers

Attention helps a model decide which words are more important.

In [ ]:
words = ["not", "a", "good", "movie"]
attention_scores = np.array([0.45, 0.05, 0.40, 0.10])

for word, score in zip(words, attention_scores):
    print(word, "->", score)

## 3. Softmax Converts Scores into Weights

Softmax turns raw scores into values that add up to 1.

In [ ]:
def softmax(scores):
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / exp_scores.sum()

raw_scores = np.array([2.0, 0.5, 1.8, 0.2])
weights = softmax(raw_scores)

for word, raw_score, weight in zip(words, raw_scores, weights):
    print(word, "raw score:", raw_score, "attention weight:", round(float(weight), 4))
print("Total weight:", round(float(weights.sum()), 4))

## 4. Tiny Self-Attention Demo

In self-attention, each word can look at other words in the same sentence.

In [ ]:
word_vectors = np.array([
    [1.0, 0.2, 0.1],
    [0.1, 0.1, 0.1],
    [0.9, 0.8, 0.2],
    [0.2, 0.3, 1.0]
])

scores = word_vectors @ word_vectors.T
attention_weights = np.apply_along_axis(softmax, 1, scores)
context_vectors = attention_weights @ word_vectors

print("Attention weights:")
print(np.round(attention_weights, 3))
print("Context vectors:")
print(np.round(context_vectors, 3))

## 5. Build a Tiny Transformer Text Classifier

This is not a full BERT model. It is a small beginner model that uses multi-head attention.

In [ ]:
texts = np.array([
    "this movie is very good",
    "excellent film and great acting",
    "i loved this story",
    "clear useful explanation",
    "this movie is very bad",
    "boring film and weak acting",
    "i hated this story",
    "confusing poor explanation"
])
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0])

text_vectorizer = layers.TextVectorization(max_tokens=1000, output_sequence_length=8, output_mode="int")
text_vectorizer.adapt(texts)
vectorized_texts = text_vectorizer(texts)
print(vectorized_texts.numpy())

In [ ]:
class TinyTransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.feed_forward = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, inputs):
        attention_output = self.attention(inputs, inputs)
        x = self.norm1(inputs + attention_output)
        ff_output = self.feed_forward(x)
        return self.norm2(x + ff_output)

In [ ]:
VOCAB_SIZE = 1000
MAX_LENGTH = 8
EMBED_DIM = 16

model = models.Sequential([
    layers.Input(shape=(MAX_LENGTH,)),
    layers.Embedding(VOCAB_SIZE, EMBED_DIM),
    TinyTransformerBlock(embed_dim=EMBED_DIM, num_heads=2, ff_dim=32),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train the Tiny Transformer

This dataset is tiny, so this is only for learning the workflow.

In [ ]:
history = model.fit(vectorized_texts, labels, epochs=40, verbose=0)
print("Final training accuracy:", round(history.history["accuracy"][-1], 4))

## 7. Predict with the Tiny Transformer

In [ ]:
def predict_text(text):
    vector = text_vectorizer(np.array([text]))
    probability = float(model.predict(vector, verbose=0)[0][0])
    label = "positive" if probability >= 0.5 else "negative"
    return {"text": text, "prediction": label, "positive_probability": round(probability, 4)}

for text in ["great useful movie", "boring weak story"]:
    print(predict_text(text))

## 8. Beginner BERT Summary

BERT is a transformer-based model.

Important beginner points:
- BERT reads text in both directions.
- BERT is pretrained on a very large amount of text.
- BERT can be fine-tuned for classification, question answering, and search.
- This notebook uses a tiny transformer idea, not a full BERT model.

## Beginner Revision Questions

1. What problem does attention solve?
2. Why do attention weights add up to 1?
3. What is multi-head attention?
4. How is a transformer different from an RNN?
5. Why is BERT called bidirectional?